# PDF Extraction

In [1]:
# Teste PDF Extraction
import sys
from pathlib import Path
def _add_src_to_path():
    from pathlib import Path
    import sys
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for candidate in candidates:
        src_path = candidate / "src"
        package_path = src_path / "byeias"
        if package_path.exists():
            src_str = str(src_path)
            if src_str not in sys.path:
                sys.path.insert(0, src_str)
            return src_path
    raise RuntimeError("Konnte den Projektpfad nicht finden. Erwartet wurde ein Ordner 'src/byeias'.")
_add_src_to_path()
from byeias.backend.extraction.text_extracter import PDFTextExtractor

# Beispiel-PDF (muss existieren)
pdf_path = "../../../../data/Businessplan_Halo_AktuellerStand.pdf"

extractor = PDFTextExtractor(language="german")
# Prüfe, ob die PDF-Datei existiert, bevor extrahiert wird

pdf_file = Path(pdf_path)
if not pdf_file.exists():
    print(f"Datei nicht gefunden: {pdf_file.resolve()}")
    sentences = []
else:
    sentences = extractor.extract_sentences(pdf_path)
    
    print(f"Extrahierte Sätze: {len(sentences)}")
for i, satz in enumerate(sentences[:5], 1):
    print(f"{i}: {satz}")

Datei nicht gefunden: C:\Users\david\OneDrive\Desktop\Byeias\data\Businessplan_Halo_AktuellerStand.pdf


# Classification

In [3]:
import logging
from pathlib import Path

import pandas as pd

def _add_src_to_path():
    """Findet den Projektroot und fügt <root>/src zu sys.path hinzu."""
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]

    for candidate in candidates:
        src_path = candidate / "src"
        package_path = src_path / "byeias"
        if package_path.exists():
            src_str = str(src_path)
            if src_str not in sys.path:
                sys.path.insert(0, src_str)
            return src_path

    raise RuntimeError(
        "Konnte den Projektpfad nicht finden. Erwartet wurde ein Ordner 'src/byeias'."
    )

SRC_PATH = _add_src_to_path()
print(f"Nutze Python-Pfad: {SRC_PATH}")

# Importiere BiasDetectionPipeline aus dem richtigen Modulpfad
from byeias.backend.classification.model_bias import BiasDetectionPipeline  # noqa: E402

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

Nutze Python-Pfad: C:\Users\david\OneDrive\Desktop\Byeias\src


In [4]:
import pandas as pd

def build_tiny_datasets():
    """Very small datasets to quickly test the pipeline functionality."""
    
    # Sexism Training Dataset
    train_df_sexism = pd.DataFrame(
        {
            "context": [
                "The team meeting was about leadership roles.",
                "New projects were discussed in the office.",
                "The application was about qualifications.",
                "Grades were distributed at school.",
                "A discussion about household chores."
            ],
            "text": [
                "Women are too emotional for leadership positions.",
                "Everyone collaborated constructively.",
                "Men are better suited for technical jobs.",
                "The class learned together fairly.",
                "Taking care of children is a woman's job."
            ],
            "sexism_label": [1, 0, 1, 0, 1], # Adjusted to 5 labels
        }
    )

    # Racism Training Dataset
    train_df_racism = pd.DataFrame(
        {
            "context": [
                "During a discussion about migration.",
                "A festival took place in the city park.",
                "The lesson was about cultural diversity.",
                "Many nations were represented at the sports event.",
                "Talking about neighborhood safety."
            ],
            "text": [
                "People from this country are all criminals.",
                "All families celebrated together.",
                "Foreigners never want to integrate.",
                "The tournament was respectful and fair.",
                "They don't belong in our neighborhood."
            ],
            "racism_label": [1, 0, 1, 0, 1], # Adjusted to 5 labels
        }
    )

    # Sexism Validation Dataset
    val_df_sexism = pd.DataFrame(
        {
            "context": [
                "Performance was evaluated in the job interview.",
                "Everyone in the club planned the event together.",
            ],
            "text": [
                "Women should stay out of the executive suite.",
                "Tasks were distributed regardless of gender.",
            ],
            "sexism_label": [1, 0],
        }
    )

    # Racism Validation Dataset
    val_df_racism = pd.DataFrame(
        {
            "context": [
                "The canteen conversation was about origins.",
                "Examples from many countries were shown in the seminar.",
            ],
            "text": [
                "This ethnicity is fundamentally lazy.",
                "Different backgrounds enrich the discussion.",
            ],
            "racism_label": [1, 0],
        }
    )

    return train_df_sexism, train_df_racism, val_df_sexism, val_df_racism

In [5]:
def build_test_samples(n=5):
    """Generates n test samples for prediction."""
    samples = [
        {
            "context": "The topic in the classroom was mathematics.",
            "text": "Girls are naturally worse at math.",
        },
        {
            "context": "A new team was formed in the company.",
            "text": "The collaboration was professional and respectful.",
        },
        {
            "context": "In a debate about immigration.",
            "text": "All migrants are a burden.",
        },
        {
            "context": "Many cultures were represented at the city festival.",
            "text": "The festival was open and inclusive.",
        },
        {
            "context": "Roles were assigned in the project meeting.",
            "text": "Women should rather take on supporting tasks.",
        },
    ]
    return samples[:n]

In [6]:
n_predictions = 5
train_df_sexism, train_df_racism, val_df_sexism, val_df_racism = build_tiny_datasets()
test_samples = build_test_samples(n=n_predictions)
test_contexts = [sample["context"] for sample in test_samples]
test_texts = [sample["text"] for sample in test_samples]

print(f"Train Sexism: {len(train_df_sexism)} | Train Racism: {len(train_df_racism)}")
print(f"Val Sexism: {len(val_df_sexism)} | Val Racism: {len(val_df_racism)}")
print(f"Inference Beispiele: {len(test_texts)}")

Train Sexism: 5 | Train Racism: 5
Val Sexism: 2 | Val Racism: 2
Inference Beispiele: 5


In [7]:
logger.info("Initialisiere Pipeline...")
pipeline = BiasDetectionPipeline(model_name="microsoft/deberta-v3-small")
print("Pipeline initialisiert")

INFO:__main__:Initialisiere Pipeline...
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/deberta-v3-small/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/deberta-v3-small/a36c739020e01763fe789b4b85e2df55d6180012/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/deberta-v3-small/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/deberta-v3-small/a36c739020e01763fe789b4b85e2df55d6180012/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/deberta-v3-small/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/deberta-v3-small/tree/main?recursive=true&expand=false "H

Pipeline initialisiert


INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/deberta-v3-small "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/deberta-v3-small/commits/main "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/deberta-v3-small/discussions?p=0 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/deberta-v3-small/commits/refs%2Fpr%2F4 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/deberta-v3-small/resolve/refs%2Fpr%2F4/model.safetensors.index.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/deberta-v3-small/resolve/refs%2Fpr%2F4/model.safetensors "HTTP/1.1 302 Found"


In [8]:
logger.info("Starte kurzes Training...")
pipeline.train(
    train_df_sexism=train_df_sexism,
    train_df_racism=train_df_racism,
    val_df_sexism=val_df_sexism,
    val_df_racism=val_df_racism,
    epochs=1,
    batch_size=2,
    lr=2e-5,
    )
print("Training abgeschlossen")

INFO:__main__:Starte kurzes Training...
Parameter 'function'=<bound method BiasDetectionPipeline._tokenize_fn of <byeias.backend.classification.model_bias.BiasDetectionPipeline object at 0x000002A127B2CEC0>> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only shown once. Subsequent hashing failures won't be shown.
Epoch 1/1 [Train]: 100%|██████████| 5/5 [00:21<00:00,  4.37s/it, loss=1.0968]


Training abgeschlossen


In [10]:
# --- VORHERSAGE (Inference) ---
logger.info("Starte Vorhersage...")
predictions = pipeline.predict(context_texts=test_contexts, target_texts=test_texts)
print(f"Vorhersagen erstellt: {len(predictions)}")

INFO:__main__:Teste Inference auf n Beispielen...


Vorhersagen erstellt: 5


In [11]:
print("\n--- INFERENCE ERGEBNISSE ---")
for idx, res in enumerate(predictions, start=1):
    print(f"Beispiel {idx}:")
    print(f"Kontext: '{res['context']}'")
    print(f"Text:    '{res['text']}'")
    print(f" -> Sexismus: {'JA' if res['sexism_prediction'] == 1 else 'NEIN'}")
    print(f" -> Rassismus: {'JA' if res['racism_prediction'] == 1 else 'NEIN'}\n")


--- INFERENCE ERGEBNISSE ---
Beispiel 1:
Kontext: 'The topic in the classroom was mathematics.'
Text:    'Girls are naturally worse at math.'
 -> Sexismus: NEIN
 -> Rassismus: NEIN

Beispiel 2:
Kontext: 'A new team was formed in the company.'
Text:    'The collaboration was professional and respectful.'
 -> Sexismus: JA
 -> Rassismus: NEIN

Beispiel 3:
Kontext: 'In a debate about immigration.'
Text:    'All migrants are a burden.'
 -> Sexismus: JA
 -> Rassismus: NEIN

Beispiel 4:
Kontext: 'Many cultures were represented at the city festival.'
Text:    'The festival was open and inclusive.'
 -> Sexismus: JA
 -> Rassismus: NEIN

Beispiel 5:
Kontext: 'Roles were assigned in the project meeting.'
Text:    'Women should rather take on supporting tasks.'
 -> Sexismus: NEIN
 -> Rassismus: NEIN



# Clasification 2

In [33]:
# --- CLASSIFICATION (Laden statt Trainieren) ---
import logging
import os
from pathlib import Path
import pandas as pd

# (Deine _add_src_to_path Funktion hier lassen...)
SRC_PATH = _add_src_to_path()
from byeias.backend.classification.model_bias import BiasDetectionPipeline
from byeias.backend.config_loader import get_backend_config

# Lade Konfiguration, um den Pfad zu den Gewichten zu erhalten
config = get_backend_config()
weights_path = config.classification.best_model_path

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Initialisiere die Pipeline
pipeline = BiasDetectionPipeline(model_name=config.classification.model_name)

# --- MODELL LADEN MIT FEHLER-ANALYSE ---
if Path(weights_path).exists():
    logger.info(f"Lade existierende Gewichte von: {weights_path}")
    try:
        # Hier bleibt das Skript aktuell stecken
        pipeline.load_model(weights_path)
        print("Modell erfolgreich geladen!")
    except Exception as e:
        print("\n" + "="*50)
        print(f"KRITISCHER FEHLER BEIM LADEN DER GEWICHTE:")
        print(e)
        print("="*50 + "\n")
else:
    print(f"WARNUNG: Gewichtsdatei nicht gefunden unter {weights_path}")
# Test-Daten vorbereiten
test_samples = build_test_samples(n=5)
test_contexts = [sample["context"] for sample in test_samples]
test_texts = [sample["text"] for sample in test_samples]

# --- VORHERSAGE (Inference) ---
logger.info("Starte Vorhersage...")
predictions = pipeline.predict(context_texts=test_contexts, target_texts=test_texts)

print("\n--- INFERENCE ERGEBNISSE ---")
for idx, res in enumerate(predictions, start=1):
    print(f"Beispiel {idx}:")
    print(f"Kontext: '{res['context']}'")
    print(f"Text:    '{res['text']}'")
    print(f" -> Sexismus: {'JA' if res['sexism_prediction'] == 1 else 'NEIN'}")
    print(f" -> Rassismus: {'JA' if res['racism_prediction'] == 1 else 'NEIN'}\n")

INFO:httpx:HTTP Request: HEAD https://huggingface.co/roberta-base/resolve/main/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/roberta-base/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/roberta-base/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/FacebookAI/roberta-base/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/roberta-base/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/FacebookAI/roberta-base/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/roberta-base/resolve/main/config.json "HTTP/1.1 200 OK"
Loading weights: 100


KRITISCHER FEHLER BEIM LADEN DER GEWICHTE:
Error(s) in loading state_dict for MultiTaskRoberta:
	Missing key(s) in state_dict: "encoder.embeddings.token_type_embeddings.weight", "encoder.embeddings.position_embeddings.weight", "encoder.encoder.layer.0.attention.self.query.weight", "encoder.encoder.layer.0.attention.self.query.bias", "encoder.encoder.layer.0.attention.self.key.weight", "encoder.encoder.layer.0.attention.self.key.bias", "encoder.encoder.layer.0.attention.self.value.weight", "encoder.encoder.layer.0.attention.self.value.bias", "encoder.encoder.layer.1.attention.self.query.weight", "encoder.encoder.layer.1.attention.self.query.bias", "encoder.encoder.layer.1.attention.self.key.weight", "encoder.encoder.layer.1.attention.self.key.bias", "encoder.encoder.layer.1.attention.self.value.weight", "encoder.encoder.layer.1.attention.self.value.bias", "encoder.encoder.layer.2.attention.self.query.weight", "encoder.encoder.layer.2.attention.self.query.bias", "encoder.encoder.layer.2

# LLM Communication

In [17]:
# Teste LLM Communication
from pathlib import Path

def _add_src_to_path():
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for candidate in candidates:
        src_path = candidate / "src"
        package_path = src_path / "byeias"
        if package_path.exists():
            src_str = str(src_path)
            if src_str not in sys.path:
                sys.path.insert(0, src_str)
            return src_path
    raise RuntimeError("Konnte den Projektpfad nicht finden. Erwartet wurde ein Ordner 'src/byeias'.")

_add_src_to_path()
from byeias.backend.llm_explanation.llm_communicator import LLMCommunicator

# Beispiel-Kontext
context_before = "Im Klassenzimmer ging es um Mathematik."
flagged_sentence = "Mädchen sind von Natur aus schlechter in Mathe."
context_after = "Die Lehrerin erklärte die nächste Aufgabe."

llm = LLMCommunicator()
result = llm.explain_bias(context_before, flagged_sentence, context_after)
print("LLM-Ergebnis:")
print(result)

INFO:httpx:HTTP Request: POST https://api.mistral.ai/v1/chat/completions "HTTP/1.1 200 OK"


LLM-Ergebnis:
{'bias_type': 'Gender ability stereotype', 'explanation': 'The sentence perpetuates the stereotype that girls inherently possess lesser mathematical ability than boys, which is not supported by scientific evidence. This reinforces harmful biases about gender and academic performance, potentially discouraging girls from engaging in STEM fields.', 'rewrite_suggestion': 'Alle Schüler:innen können mit der richtigen Förderung und Übung gute Leistungen in Mathe erzielen.'}


# Complete Pipeline

In [34]:
import re

def process_data(input_text):
    raw_sentences = re.split(r'(?<=[.!?])\s+', input_text.strip())
    sentences = [s.strip() for s in raw_sentences if s.strip()]

    print(f"Eingabetext in {len(sentences)} Sätze aufgeteilt.")

    if not sentences:
        print("Keine Sätze zum Verarbeiten gefunden.")
        return []

    pipeline = BiasDetectionPipeline(model_name=config.classification.model_name)
    llm = LLMCommunicator()

    context_texts = []
    target_texts = []
    contexts_before = []
    contexts_after = []

    for i in range(len(sentences)):
        target = sentences[i]
        
        before = sentences[i-1] if i > 0 else ""
        
        after = sentences[i+1] if i + 1 < len(sentences) else ""
        
        combined_context = f"{before} {after}".strip()
        
        target_texts.append(target)
        context_texts.append(combined_context)
        
        contexts_before.append(before)
        contexts_after.append(after)

    print("Starte Vorhersage für alle Sätze...")
    inference_results = pipeline.predict(
        context_texts=context_texts, 
        target_texts=target_texts
    )

    print("Inferenz-Ergebnisse:", inference_results)

    explanations = []

    for i, result in enumerate(inference_results):
        if result["sexism_prediction"] == 1 or result["racism_prediction"] == 1:
            print(f"\n[BIAS GEFUNDEN] in Satz {i+1}: '{result['text']}'")
            
            explanation = llm.explain_bias(
                context_before=contexts_before[i],
                flagged_sentence=result["text"],
                context_after=contexts_after[i],
            )
            
            explanations.append({
                "satz_index": i + 1,
                "geflaggter_satz": result["text"],
                "bias_typ": "Sexismus" if result["sexism_prediction"] == 1 else "Rassismus",
                "llm_erklaerung": explanation
            })

    return explanations

# --- TEST ---
text = "The Topic is about payment. Girls get paid less. The teacher explained the next task. Everyone was listening."
ergebnisse = process_data(text)

print("\n--- FINALE ZUSAMMENFASSUNG ---")
for erg in ergebnisse:
    print(f"Satz {erg['satz_index']} ({erg['bias_typ']}): {erg['geflaggter_satz']}")
    print(f"Erklärung: {erg['llm_erklaerung']}\n")

Eingabetext in 4 Sätze aufgeteilt.


INFO:httpx:HTTP Request: HEAD https://huggingface.co/roberta-base/resolve/main/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/roberta-base/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/roberta-base/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/FacebookAI/roberta-base/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/roberta-base/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/FacebookAI/roberta-base/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/roberta-base/resolve/main/config.json "HTTP/1.1 200 OK"
Loading weights: 100

Starte Vorhersage für alle Sätze...
Inferenz-Ergebnisse: [{'context': 'Girls get paid less.', 'text': 'The Topic is about payment.', 'sexism_prediction': 1, 'racism_prediction': 1}, {'context': 'The Topic is about payment. The teacher explained the next task.', 'text': 'Girls get paid less.', 'sexism_prediction': 1, 'racism_prediction': 1}, {'context': 'Girls get paid less. Everyone was listening.', 'text': 'The teacher explained the next task.', 'sexism_prediction': 1, 'racism_prediction': 1}, {'context': 'The teacher explained the next task.', 'text': 'Everyone was listening.', 'sexism_prediction': 1, 'racism_prediction': 1}]

[BIAS GEFUNDEN] in Satz 1: 'The Topic is about payment.'


INFO:httpx:HTTP Request: POST https://api.mistral.ai/v1/chat/completions "HTTP/1.1 200 OK"



[BIAS GEFUNDEN] in Satz 2: 'Girls get paid less.'


...


[BIAS GEFUNDEN] in Satz 3: 'The teacher explained the next task.'


INFO:httpx:HTTP Request: POST https://api.mistral.ai/v1/chat/completions "HTTP/1.1 200 OK"



[BIAS GEFUNDEN] in Satz 4: 'Everyone was listening.'


INFO:httpx:HTTP Request: POST https://api.mistral.ai/v1/chat/completions "HTTP/1.1 200 OK"



--- FINALE ZUSAMMENFASSUNG ---
Satz 1 (Sexismus): The Topic is about payment.
Erklärung: {'bias_type': 'gendered_assumption', 'explanation': "The highlighted sentence itself is neutral, but the following context ('Girls get paid less') introduces a gender bias by implying that only girls or women are affected by pay disparities. This could reinforce the stereotype that pay inequality is exclusively a female issue, rather than a systemic problem affecting all genders. The phrasing also risks oversimplifying the topic by framing it as a binary issue.", 'rewrite_suggestion': 'The topic is about payment disparities and how they affect different groups.'}

Satz 2 (Sexismus): Girls get paid less.
Erklärung: {'bias_type': 'Gender-based generalization', 'explanation': 'The sentence implies a universal truth about all girls being paid less, which reinforces the stereotype that women or girls are inherently disadvantaged in payment without acknowledging systemic or contextual factors. It also o